<a href="https://colab.research.google.com/github/Yashraj-mlrobo/flyrank-task-1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yashraj-mlrobo/flyrank-task-1/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Ranking / Prioritization Task (implemented via a Scoring System)


In [ ]:
'''


Task Type: Scoring (which directly enables Ranking)

I am building an Opportunity Scoring engine to support a manual content refresh workflow. While a Classification task could flag a page as simply "degraded" (1) or "healthy" (0), a binary flag doesn't help an editorial team know where to start when faced with thousands of degraded pages.

By framing this as a Scoring task, I can assign a continuous opportunity or utility value to each URL based on its structural characteristics. We then use those scores to output a Ranked queue, ensuring editors always tackle the highest-yield, most critical URLs at the very top of their list.

'''


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
'''
What I will predict: A continuous Opportunity Score representing a URL's absolute traffic and CTR degradation relative to its position group baseline.

Where the label comes from: The baseline targets are derived from observed outcomes in the data rather than arbitrary manual rules.
                            We calculate the true, historical median performance behavior for distinct position cohorts (for example, our observed top-tier performance baseline).
                            The actual target label or delta is then computed as the mathematical distance between a specific URL's observed performance and its cohort's expected baseline, flagging deviations where a page is significantly underperforming compared to historical realities.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
"""
Evaluation Metric: Precision@20 (P@20)

good?: A P@20 score of greater than or equal to 80% (meaning at least 16 out of the top 20 surfaced URLs are genuinely high-yield, degraded opportunities).

The given operational constraint is a highly constrained, manual human editorial workflow.
The primary bottleneck is editor time; therefore, our biggest resource drain is a False Positive (sending an editor to overhaul a page that doesn't actually yield a traffic lift).
Because the workflow handles recommendations in small batches, maximizing precision at the very top of the ranked queue is far more critical than capturing every single degraded URL across the site (Recall).
If 80% or more of the top 20 recommendations consistently translate into valid, high-impact content actions, the system successfully eliminates wasted editorial overhead.
Thus a P@20 precision metric is our sucess metric

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
import os
import pandas as pd
import numpy as np

# fetching through through established colab pipeline
csv_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(csv_path):
    import urllib.request
    os.makedirs("data/raw", exist_ok=True)
    url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
    urllib.request.urlretrieve(url, csv_path)

df = pd.read_csv(csv_path)

# 2. Filter for active tracking records
active_df = df[df['avg_position'] > 0].copy()

# 3. Sketch the dynamic target column based on our performance baseline gap
ctr_threshold = active_df['ctr'].quantile(0.25)
active_df['target_opportunity_flag'] = (active_df['ctr'] < ctr_threshold).astype(int)

# 4. Display the clean unit of analysis preview
unit_of_analysis_preview = active_df[['content_id', 'avg_position', 'ctr', 'target_opportunity_flag']].head(5)

print("--- UNIT OF ANALYSIS PREVIEW (1 Row = 1 Unique URL) ---")
display(unit_of_analysis_preview)
print("-------------------------------------------------------")

--- UNIT OF ANALYSIS PREVIEW (1 Row = 1 Unique URL) ---


,content_id,avg_position,ctr,target_opportunity_flag
0,content_304f48230142,10.6,0.76,0
1,content_a1fb4e703a9e,20.3,0.05,0
2,content_9aa793d4d895,36.5,0.09,0
3,content_331d6c4de07b,6.2,0.49,0
4,content_d99b7a2d90ca,44.0,0.13,0


-------------------------------------------------------


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
"""
Our initial data validation proved that simple heuristic thresholds are mathematically blind to true performance degradation.
For example, setting a static rule like if avg_position > 0.55 provides zero operational utility because the observed mean CTR remains virtually identical at 5.87% versus 5.88% across that threshold.

Content performance degradation is fundamentally non-linear and multi-variant. A page's true traffic potential isn't dictated by a single metric; it is a complex intersection of search volume scale, impression velocity over time (impressions_last_30d vs. impressions_prev_30d), word count tiers, and historical baseline deviations within specific intent cohorts.
Surfacing high-yield refresh opportunities requires mapping this multidimensional space simultaneously.

A hardcoded if-statement can only draw rigid, flat lines that either flood the editorial queue with false positives or completely miss subtle, high-value decay trends that a trained scoring model naturally surfaces.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.